[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/03b_mobo.ipynb)

# 03b — Multi-objective Bayesian optimization

**Purpose.** Search the 36-dimensional tilt space with Ax, optimizing all five
KPIs jointly, and report one configuration to deploy.

**Inputs.** The committed cell table in `configs/simulation.yaml`, the scenario
manifest, and `data/processed/mdt.parquet` for the band priority score.

**Outputs.** `outputs/optim/mobo/<timestamp>/` — the evaluation history, the
Pareto front, the tilt table, the winner's radio map, and a run document.

---

### What this is, and is not

Plain multi-objective BO: a Gaussian-process model per KPI and a hypervolume
acquisition, with **no trust region**. That is a deliberate departure from the
TuRBO the README names, recorded in
[ADR 0002](../docs/adr/0002-bayesian-optimization-without-a-trust-region.md)
— an evaluation here costs roughly 30-40 s, so a few hundred cost hours rather
than days, and evaluations are not the scarce resource a trust region exists to
conserve.

The five KPIs and their priority order come from
[ADR 0001](../docs/adr/0001-five-kpis-under-lexicographic-priority.md) and are
defined only in `src/kpi/`. This notebook never re-derives a threshold.

**Baselines live in `03a_baseline.ipynb`.** Run that one too — a BO result with
nothing to compare it against says very little.


## 0. Environment

Run this section first, wherever you are.

**Locally** it walks up to the project root and makes it the working directory,
so the root-relative paths in the configs resolve the way they do for
`task bo` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path`, and installs
what Colab does not ship. Sionna-RT needs a CUDA GPU — select a GPU runtime
first, or every evaluation will fail.


In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook in this project.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = ""  # the project root is the repository root

# (import name, pip name). Colab ships numpy, pandas, pyarrow, matplotlib and
# seaborn; these are the ones it does not. sionna-rt needs a CUDA runtime, so
# select a GPU runtime before running this notebook.
COLAB_PACKAGES = [
    ("hydra", "hydra-core"),
    ("sionna.rt", "sionna-rt"),
    ("ax", "ax-platform"),
]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR if SUBDIR else checkout
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Reading them
# from the environment lets a smoke run shrink the budget, or a Colab session
# repoint the paths, without editing the notebook.
CONFIG_OVERRIDES: list[str] = os.environ.get("BAND_TILT_OVERRIDES", "").split()

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ is DVC-tracked and not part of the clone, so a fresh Colab runtime has
# no scenario to optimize against. Either run notebook 00 first, or mount Drive
# and point the config at a copy that already holds one. Drive also survives a
# runtime reset, which /content does not.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# PROJECT_ROOT = "/content/drive/MyDrive/band-tilt"
# CONFIG_OVERRIDES += [
#     f"simulation.output.manifest_file={PROJECT_ROOT}/data/external/scenario.json",
#     f"data.output.mdt_file={PROJECT_ROOT}/data/processed/mdt.parquet",
#     f"bo.output.dir={PROJECT_ROOT}/outputs/optim",
# ]

## 1. Setup

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

# Ax logs every generated trial at INFO, which here means all 36 tilts per
# trial. Warnings still come through - short batches are worth seeing.
logging.getLogger("ax").setLevel(logging.WARNING)

cfg = load_config(overrides=["bo.method=mobo", *CONFIG_OVERRIDES])
set_seed(cfg.seed)
setup_plotting()

pd.set_option("display.max_columns", 60)
print(f"method={cfg.bo.method}  seed={cfg.bo.seed}")
print(f"budget: n_init={cfg.bo.budget.n_init} n_iter={cfg.bo.budget.n_iter} "
      f"batch={cfg.bo.budget.batch_size}   rule: {cfg.bo.rule.n_steps} steps "
      f"x {cfg.bo.rule.n_rounds} rounds")

## 2. The search space

One absolute tilt per **(cell, band)** pair. The bounds are the committed ones
in `configs/simulation.yaml`; nothing here may widen them, and
`TiltSpace.to_cells` raises rather than ray-tracing a proposal that leaves the
box.


In [ ]:
from src.optim.space import TiltSpace

space = TiltSpace.from_config(cfg)
print(f"{space.n_dim} decision variables: {len(space.cells)} cells x {len(space.band_names)} bands")

bounds = space.as_frame(space.baseline)
bounds.groupby("band", observed=True).agg(
    baseline=("tilt_deg", "first"),
    low=("tilt_min_deg", "first"),
    high=("tilt_max_deg", "first"),
    n_cells=("cell", "size"),
)

## 3. The incumbent

Everything is measured against the tilts currently committed in
`configs/simulation.yaml`. Evaluating them first does three jobs: it is the
reference row of every comparison, it anchors Ax's objective thresholds, and it
is the check that this evaluator agrees with the simulation stage — these five
numbers must match the ones `data/interim/radio_map.npz` scores to.


In [ ]:
from src.kpi.serving import max_rsrp
from src.optim.evaluator import Evaluator
from src.optim.objective import KPI_NAMES


def kpi_table(named):
    """One column per configuration, one row per KPI, in priority order."""
    return pd.DataFrame({name: kpi.as_dict() for name, kpi in named.items()}).loc[list(KPI_NAMES)]

In [ ]:
evaluator = Evaluator(cfg, keep_rsrp=True)
print(f"scenario {evaluator.scenario_id}   bands {evaluator.band_labels}")

incumbent = evaluator.evaluate(space.baseline)
print(f"one evaluation = {incumbent.seconds:.1f}s of ray tracing\n")
kpi_table({"incumbent": incumbent.kpi})

## 4. The run

`ax_search` is the same loop random search uses; only the generation strategy
differs. Ax spends `budget.n_init` trials on Sobol, then switches to the model.

**This is the slow cell — budget hours, not minutes.** Ray tracing costs roughly
30-40 s per candidate. Fitting five GPs and optimizing a five-objective
hypervolume acquisition costs little at first and grows with the observation
count, overtaking the simulator somewhere past a hundred evaluations. The split
is printed below so you can see which half you are paying for.


In [ ]:
import time

from src.optim.search import run_search

started = time.time()
history = run_search(evaluator, cfg, "mobo")
wall_clock = time.time() - started

frame = history.frame()
ray_tracing = frame["seconds"].sum()
print(f"{len(history)} evaluations in {wall_clock / 60:.1f} min")
print(f"  ray tracing {ray_tracing / 60:.1f} min ({ray_tracing / wall_clock:.0%})")
print(f"  model       {(wall_clock - ray_tracing) / 60:.1f} min")
print(f"  non-dominated: {int(frame['on_pareto'].sum())}")
frame["phase"].value_counts()

## 5. Progress

Hypervolume is anchored on the incumbent, so it measures improvement over what
is deployed and a configuration worse than the incumbent contributes nothing.
Expect very small absolute numbers — it is a five-way product of fractional
gains — so read the trend, not the value, and note the log scale.


In [ ]:
from src.optim.objective import hypervolume_trace

trace = hypervolume_trace(history.kpis, incumbent.kpi)
switch = frame.index[frame["phase"] == "search"].min()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(trace, marker=".", lw=1)
axes[0].set_yscale("symlog", linthresh=1e-12)
axes[0].axvline(switch, color="0.6", ls="--", lw=1)
axes[0].annotate("model takes over", (switch, axes[0].get_ylim()[1]),
                 xytext=(4, -12), textcoords="offset points", fontsize=9, color="0.4")
axes[0].set(xlabel="evaluation", ylabel="hypervolume over incumbent",
            title="Progress")

# Best-so-far per KPI, each in its own direction. Four fall, one rises.
for name in KPI_NAMES:
    series = frame[name]
    running = series.cummax() if name == "band_priority_score" else series.cummin()
    axes[1].plot(running / abs(series.iloc[0]), label=name, lw=1.4)
axes[1].axvline(switch, color="0.6", ls="--", lw=1)
axes[1].axhline(1.0, color="0.8", lw=1)
axes[1].set(xlabel="evaluation", ylabel="best so far / incumbent",
            title="Each KPI, relative to the incumbent")
axes[1].legend(fontsize=8)
fig.tight_layout()

## 6. The front

Five objectives cannot be plotted at once, so two views: the trade-off that
matters most here, and parallel coordinates over the whole front.

Band priority score starts near its floor — almost every covered UE is served by
the 700 MHz layer, which is what the band that propagates furthest will do. It
is therefore the easiest KPI to move, and buying it means tilting the low band
down until it stops serving, which is exactly how holes open. That trade is the
interesting part of this front.


In [ ]:
pareto = history.pareto_frame()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].scatter(frame["hole_rate"], frame["band_priority_score"], s=18, c="0.75",
                label="evaluated")
axes[0].scatter(pareto["hole_rate"], pareto["band_priority_score"], s=42,
                c="tab:blue", label="non-dominated")
axes[0].scatter([incumbent.kpi.hole_rate], [incumbent.kpi.band_priority_score],
                s=130, marker="*", c="tab:red", zorder=5, label="incumbent")
axes[0].set(xlabel="hole rate (minimise)", ylabel="band priority score (maximise)",
            title="The trade the front is made of")
axes[0].legend(fontsize=8)

# Parallel coordinates, each KPI scaled to its own observed range so five
# incomparable units share an axis.
scaled = frame[list(KPI_NAMES)]
scaled = (scaled - scaled.min()) / (scaled.max() - scaled.min()).replace(0, 1)
for row in scaled[~frame["on_pareto"]].to_numpy():
    axes[1].plot(row, color="0.85", lw=0.7, zorder=1)
for row in scaled[frame["on_pareto"]].to_numpy():
    axes[1].plot(row, color="tab:blue", lw=1.2, alpha=0.8, zorder=2)
axes[1].plot(scaled.iloc[0].to_numpy(), color="tab:red", lw=2.4, zorder=3, label="incumbent")
axes[1].set_xticks(range(len(KPI_NAMES)))
axes[1].set_xticklabels([n.replace("_", "\n") for n in KPI_NAMES], fontsize=8)
axes[1].set(ylabel="scaled to observed range", title=f"Pareto front ({len(pareto)} points)")
axes[1].legend(fontsize=8)
fig.tight_layout()

## 7. Choosing one configuration

The deliverable is one tilt table, not a front. ADR 0001's priority order picks
it: hole rate first, and a difference smaller than that KPI's tolerance in
`configs/kpi.yaml` is a tie that passes the decision down the list.

It is a single pass, not a sort. With slack, the relation is not transitive — a
can tie b and b tie c while a beats c — so there is no ordering to find, and a
"ranking" of candidates would depend on the order they arrived in.


In [ ]:
best_index = history.best_index(cfg)
best = history.results[best_index]

print(f"winner: evaluation {best_index} "
      f"(phase={frame.loc[best_index, 'phase']}, "
      f"node={frame.loc[best_index, 'generation_node']})\n")

comparison = kpi_table({"incumbent": incumbent.kpi, "best": best.kpi})
comparison["change"] = comparison["best"] - comparison["incumbent"]
comparison["direction"] = ["maximise" if n == "band_priority_score" else "minimise"
                           for n in comparison.index]
comparison

### How far each antenna moved

In [ ]:
table = history.tilt_table(best.tilt_deg)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for band, group in table.groupby("band", observed=True):
    axes[0].scatter(group["current_tilt_deg"], group["optimized_tilt_deg"], label=band, s=45)
limits = [table[["current_tilt_deg", "optimized_tilt_deg"]].to_numpy().min() - 1,
          table[["current_tilt_deg", "optimized_tilt_deg"]].to_numpy().max() + 1]
axes[0].plot(limits, limits, color="0.7", ls="--", lw=1)
axes[0].set(xlabel="current tilt (deg)", ylabel="optimized tilt (deg)",
            title="Where each cell-band moved")
axes[0].legend(fontsize=8, title="band")

order = table.sort_values("delta_tilt_deg")
axes[1].barh(range(len(order)), order["delta_tilt_deg"],
             color=["tab:blue" if d < 0 else "tab:orange" for d in order["delta_tilt_deg"]])
axes[1].set_yticks(range(len(order)))
axes[1].set_yticklabels(order["cell"] + " " + order["band"], fontsize=6)
axes[1].axvline(0, color="0.4", lw=1)
axes[1].set(xlabel="delta tilt (deg)", title="Movement, reported not optimized")
fig.tight_layout()

table.sort_values("delta_tilt_deg", key=abs, ascending=False).head(10)

### Coverage, before and after

In [ ]:
optimized = evaluator.evaluate(best.tilt_deg)

before = max_rsrp(incumbent.rsrp)
after = max_rsrp(optimized.rsrp)
hole_dbm = float(cfg.kpi.hole_dbm)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, data, title in ((axes[0], before, "incumbent"), (axes[1], after, "optimized")):
    im = ax.imshow(data, origin="lower", vmin=-120, vmax=-60, cmap="viridis")
    ax.set(title=f"Best-server RSRP - {title}", xticks=[], yticks=[])
    fig.colorbar(im, ax=ax, label="dBm", fraction=0.046)

# Where coverage changed state, which the RSRP panels alone do not show.
opened = (before > hole_dbm) & (after <= hole_dbm)
filled = (before <= hole_dbm) & (after > hole_dbm)
change = np.zeros_like(before)
change[filled] = 1
change[opened] = -1
axes[2].imshow(change, origin="lower", cmap="coolwarm_r", vmin=-1, vmax=1)
axes[2].set(title=f"holes filled {filled.sum()} / opened {opened.sum()}", xticks=[], yticks=[])
fig.tight_layout()

## 8. Persist

The same writer the CLI uses, so a notebook run and `task bo` leave byte-identical
artifacts. The directory is timestamped: comparing methods means comparing runs,
and a rerun must not overwrite the one it is compared against.


In [ ]:
NOTEBOOK = "03b_mobo.ipynb"

In [ ]:
from src.optim.history import LocalRunWriter, write_run
from src.optim.run import output_directory

directory = output_directory(cfg, cfg.bo.method)
writer = LocalRunWriter(directory)

# Re-solve the winner with its map retained, and archive it in the schema the
# simulation stage writes, so src/kpi and the EDA plots read it unchanged.
archived = evaluator.evaluate(best.tilt_deg)
radio_map_path = evaluator.write_radio_map(directory / "best_radio_map.npz", archived)

written = write_run(
    history,
    writer,
    cfg,
    method=cfg.bo.method,
    best_index=best_index,
    extra={
        "scenario_id": evaluator.scenario_id,
        "best_radio_map": str(radio_map_path),
        "notebook": NOTEBOOK,
    },
)
evaluator.close()  # release the scene and its GPU memory

for name, locator in written.items():
    print(f"{name:>10}: {locator}")
print(f"{"radio_map":>10}: {radio_map_path}")

## 9. Handoff checklist

- [ ] The incumbent's KPIs matched `data/interim/radio_map.npz`
- [ ] Every evaluated tilt stayed inside the committed bounds
- [ ] The five KPIs came from `src/kpi/`, with no threshold restated here
- [ ] The winner was chosen by the ADR 0001 single pass, not by eye
- [ ] `03a_baseline.ipynb` has been run, so there is something to compare against
- [ ] The run directory holds history, pareto, best_tilt, best_radio_map and run.json
